# RAS 2026 — CG v3 (Ultimate)

**v3 improvements over v2:**
1. **CG-enhanced repair**: destroyed demands re-priced over ALL hubs before OR-Tools → closes integrality gap
2. **Adaptive K**: K grows when solution stagnates, resets on improvement
3. **Hub destroy operator**: destroy ALL demands using worst hubs (structural perturbation)
4. **Perturbation + CG restart**: every N episodes, randomly perturb → mini-CG → escape basin

**Pipeline:** CG → [AISO destroy / Hub destroy / Perturb+CG] → CG-repair → SA → repeat

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE = '/content/drive/MyDrive/ras2026_cache'
os.makedirs(DRIVE_CACHE, exist_ok=True)

if not os.path.exists('ras2026'):
    !git clone https://github.com/nepersoned/ras2026.git
os.chdir('ras2026')
!git pull
print('cwd:', os.getcwd())

In [ ]:
!pip install -q pybind11 ortools scipy numpy pandas torch joblib scikit-learn

In [ ]:
DRIVE_DATA = '/content/drive/MyDrive/ras_release_v1.1'
if not os.path.exists('ras_release_v1.1'):
    os.symlink(DRIVE_DATA, 'ras_release_v1.1')
print('Data ready:', os.path.exists('ras_release_v1.1'))

## 1. Build C++

In [ ]:
import subprocess, sys

pybind_inc = subprocess.check_output(
    [sys.executable, '-m', 'pybind11', '--includes']
).decode().strip()
ext_suffix = subprocess.check_output(
    [sys.executable, '-c', 'import sysconfig; print(sysconfig.get_config_var("EXT_SUFFIX"))']
).decode().strip()

for name, src, flags in [
    ('evaluator',    'src/evaluator.cpp',    '-O3'),
    ('cg_pricer_v2', 'src/cg_pricer_v2.cpp', '-O3 -fopenmp'),
]:
    out = f'src/{name}{ext_suffix}'
    r = subprocess.run(f'g++ {flags} -shared -fPIC {pybind_inc} {src} -o {out}',
                       shell=True, capture_output=True, text=True)
    print(f'{name}: {"OK" if r.returncode==0 else "ERROR"}' +
          (f'\n{r.stdout+r.stderr}' if r.returncode else ''))

In [ ]:
import sys
sys.path.insert(0, 'src')
sys.path.insert(0, '.')

import math, random, time, json, csv
from collections import defaultdict

import numpy as np
import joblib
from sklearn.cluster import AgglomerativeClustering
from ortools.linear_solver import pywraplp
from ortools.sat.python import cp_model

import evaluator as ev
import cg_pricer_v2 as cgp

from solver import (
    load_layer, load_od_matrix, Network, Demand, Solution,
    greedy_construct, evaluate, build_json,
    COMMODITY_TO_BLOCK_TYPE, DIRECT_ONLY,
)

COMMODITY_IDX    = {'Manifest': 0, 'Bulk': 1, 'Intermodal': 2, 'Multilevel': 3}
DIRECT_ONLY_COMM = {'Intermodal', 'Automobile'}
MAX_HUBS = 50
print('Imports OK')

## 2. Environment

In [ ]:
def load_env(layer, multiplier):
    cache_jl  = f'{DRIVE_CACHE}/od_matrix_{layer}.joblib'
    cache_pkl = f'{DRIVE_CACHE}/od_matrix_{layer}.pkl'
    net_cache = f'{DRIVE_CACHE}/network_{layer}.joblib'

    nodes_df, links_df, demands_scaled, demands_raw, settings = load_layer(layer, multiplier)
    yard_rows = nodes_df[nodes_df['node_type'] == 'yard']
    yard_info = {
        int(r['node_id']): {
            'num_tracks':        float(r.get('num_tracks', 9999) or 9999),
            'handling_capacity': float(r.get('handling_capacity', 1e9) or 1e9),
            'handling_cost':     float(r.get('handling_cost', 0) or 0),
        } for _, r in yard_rows.iterrows()
    }
    origin_ids   = set(demands_scaled['origin_yard_id'].astype(int))
    dest_ids     = set(demands_scaled['dest_yard_id'].astype(int))
    all_yard_ids = sorted(origin_ids | dest_ids)

    if os.path.exists(cache_jl) and os.path.exists(net_cache):
        t0 = time.time()
        print(f'  Loading od_matrix + Network from cache...')
        od_matrix = joblib.load(cache_jl)
        net       = joblib.load(net_cache)
        print(f'  Loaded in {time.time()-t0:.1f}s')
    else:
        if os.path.exists(cache_jl):
            od_matrix = joblib.load(cache_jl)
        elif os.path.exists(cache_pkl):
            import pickle
            with open(cache_pkl, 'rb') as f: od_matrix = pickle.load(f)
            joblib.dump(od_matrix, cache_jl, compress=3)
        else:
            all_od_pairs = {(o,d) for o in all_yard_ids for d in all_yard_ids if o!=d}
            od_matrix    = load_od_matrix(all_od_pairs)
            joblib.dump(od_matrix, cache_jl, compress=3)
        print(f'  Building Network (Dijkstra)...')
        t0  = time.time()
        net = Network(nodes_df, links_df, origin_ids, dest_ids, settings, verbose=True)
        print(f'  Built in {time.time()-t0:.1f}s, caching...')
        try: joblib.dump(net, net_cache, compress=3); print('  Cached.')
        except Exception as e: print(f'  Cache failed: {e}')

    demands = [
        Demand(
            idx=idx, demand_id=int(r['demand_id']),
            commodity_type=str(r['block_type']),
            block_type=COMMODITY_TO_BLOCK_TYPE.get(str(r['block_type']), 'Manifest'),
            origin=int(r['origin_yard_id']), dest=int(r['dest_yard_id']),
            volume=int(r['volume']),
            sp_dist=od_matrix.get(
                (int(r['origin_yard_id']), int(r['dest_yard_id'])),
                net.dist(int(r['origin_yard_id']), int(r['dest_yard_id']))
            ),
        ) for idx, (_, r) in enumerate(demands_scaled.iterrows())
    ]
    return dict(net=net, od_matrix=od_matrix, settings=settings,
                yard_info=yard_info, demands=demands, all_yards=all_yard_ids,
                nodes_df=nodes_df, links_df=links_df, demands_raw=demands_raw)


def _build_cands(dem, env):
    s   = env['settings']
    tc  = float(s.get('transport_cost_coefficient', 1.0))
    ic  = float(s.get('interchange_cost', 100))
    M   = float(s.get('stress_penalty_M', 5))
    net = env['net']
    cands = [(None, M * dem.volume * dem.sp_dist)]
    od_d = net.dist(dem.origin, dem.dest)
    if not math.isinf(od_d):
        cands.append(([(dem.origin, dem.dest)],
                      tc*dem.volume*od_d + ic*dem.volume*net.interchanges(dem.origin, dem.dest)))
    if dem.commodity_type not in DIRECT_ONLY_COMM:
        for hub in env['all_yards']:
            if hub in (dem.origin, dem.dest): continue
            d1 = net.dist(dem.origin, hub)
            d2 = net.dist(hub, dem.dest)
            if math.isinf(d1) or math.isinf(d2): continue
            hc  = env['yard_info'].get(hub, {}).get('handling_cost', 0.0)
            cost = (tc*dem.volume*(d1+d2)
                    + ic*dem.volume*(net.interchanges(dem.origin,hub)+net.interchanges(hub,dem.dest))
                    + hc*dem.volume)
            cands.append(([(dem.origin, hub), (hub, dem.dest)], cost))
    return cands


def precompute(env):
    data = []
    for dem in env['demands']:
        if dem.volume <= 0:
            data.append(None); continue
        data.append({'candidates': _build_cands(dem, env),
                     'direct_only': dem.commodity_type in DIRECT_ONLY_COMM})
    return data


print('Environment ready.')

## 3. CG

In [ ]:
def cg_solve(env, dd, max_iter=50, time_limit_s=180, verbose=True):
    demands = env['demands']
    s       = env['settings']
    tc      = float(s.get('transport_cost_coefficient', 1.0))
    M_pen   = float(s.get('stress_penalty_M', 5))
    n       = len(demands)
    t0      = time.time()

    od_dist_cpp = {f'{o}_{d}': float(v)
                   for (o,d), v in env['od_matrix'].items() if math.isfinite(v)}
    yard_hc_cpp = {k: float(v['handling_cost']) for k,v in env['yard_info'].items()}
    demands_cpp = [(dem.idx, dem.origin, dem.dest, dem.volume,
                    dem.commodity_type in DIRECT_ONLY_COMM) for dem in demands]

    columns = [list(d['candidates']) if d else
               [(None, M_pen * demands[i].volume * demands[i].sp_dist)]
               for i, d in enumerate(dd)]

    best_obj, best_routings = float('inf'), None

    for it in range(max_iter):
        if time.time() - t0 > time_limit_s:
            if verbose: print(f'  CG: time limit at iter {it}')
            break

        solver = pywraplp.Solver.CreateSolver('GLOP')
        solver.SuppressOutput()
        x = {(i,ri): solver.NumVar(0,1,f'x{i}_{ri}')
             for i in range(n) for ri in range(len(columns[i]))}
        for i in range(n):
            ct = solver.Constraint(1,1,f'd{i}')
            for ri in range(len(columns[i])): ct.SetCoefficient(x[(i,ri)], 1)
        obj = solver.Objective(); obj.SetMinimization()
        for (i,ri),v in x.items(): obj.SetCoefficient(v, columns[i][ri][1])
        if solver.Solve() not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
            break

        lp_obj = solver.Objective().Value()
        duals  = [0.0]*n
        for i in range(n):
            ct = solver.LookupConstraint(f'd{i}')
            if ct: duals[i] = ct.dual_value()

        new_cols = cgp.price_full_hubs(od_dist_cpp, env['all_yards'], yard_hc_cpp,
                                       demands_cpp, duals, tc, M_pen, tol=1e-4)
        added = 0
        for nc in new_cols:
            i = nc.demand_idx
            route = ([(demands[i].origin, demands[i].dest)] if nc.hub==-1
                     else [(demands[i].origin, nc.hub), (nc.hub, demands[i].dest)])
            if not any(c[0]==route for c in columns[i]):
                columns[i].append((route, nc.cost)); added += 1

        routings = [columns[i][max(range(len(columns[i])),
                    key=lambda ri: x[(i,ri)].solution_value())][0]
                    for i in range(n)]
        if lp_obj < best_obj:
            best_obj, best_routings = lp_obj, routings

        if verbose:
            print(f'  CG {it+1:3d} | LP={lp_obj:>18,.0f} | +cols={added:5d} | {time.time()-t0:.1f}s')
        if added == 0:
            if verbose: print('  CG converged.')
            break

    for i,d in enumerate(dd):
        if d: d['candidates'] = columns[i]

    if best_routings is None:
        best_routings = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                         env['settings'], env['yard_info']).routings
    return best_routings


print('CG ready.')

## 4. C++ Solver

In [ ]:
def build_cpp_solver(env, dd, routings, verbose=False):
    demands = env['demands']
    s = env['settings']

    dem_list = [(i, dem.origin, dem.dest, dem.volume, dem.sp_dist,
                 COMMODITY_IDX.get(COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest'),0),
                 dem.commodity_type in DIRECT_ONLY_COMM)
                for i, dem in enumerate(demands)]

    cand_list = []
    for i, d in enumerate(dd):
        if d is None:
            cand_list.append([(True,False,-1,0.0,0.0,-1,-1,-1,-1)]); continue
        dem = demands[i]
        cpp_c = []
        for route, cost in d['candidates']:
            if route is None:
                cpp_c.append((True,False,-1,cost,0.0,-1,-1,-1,-1))
            elif len(route)==1:
                cpp_c.append((False,True,-1,cost,0.0,route[0][0],route[0][1],-1,-1))
            else:
                h  = route[0][1]
                hc = env['yard_info'].get(h,{}).get('handling_cost',0.0)*dem.volume
                cpp_c.append((False,False,h,cost-hc,hc,route[0][0],h,h,route[1][1]))
        cand_list.append(cpp_c)

    init_idx = []
    for i, r in enumerate(routings):
        d = dd[i]
        if d is None: init_idx.append(0); continue
        found = next((ri for ri,(route,_) in enumerate(d['candidates']) if route==r), 0)
        init_idx.append(found)

    cpp_s = ev.Settings()
    cpp_s.block_fixed_cost     = float(s.get('block_fixed_cost', 1500))
    cpp_s.unserved_penalty     = float(s.get('stress_penalty_M', 5))
    cpp_s.min_block_vol_short  = 5.0
    cpp_s.min_block_vol_medium = 10.0
    cpp_s.min_block_vol_long   = 15.0

    solver = ev.RasSolver()
    solver.init(dem_list, cand_list, init_idx,
                {k: int(v['num_tracks'])        for k,v in env['yard_info'].items()},
                {k: float(v['handling_capacity'])for k,v in env['yard_info'].items()},
                {f'{o}_{d}': float(v) for (o,d),v in env['od_matrix'].items() if math.isfinite(v)},
                cpp_s)
    if verbose: print(f'  C++ init: {solver.get_score():,.0f}')
    return solver


def get_routings(solver, dd):
    return [dd[i]['candidates'][ri][0] if dd[i] else None
            for i, ri in enumerate(solver.get_routes())]


print('C++ solver ready.')

## 5. Operators

In [ ]:
# ── AISO v2 ────────────────────────────────────────────────────────────────
def build_smart_M(mu, K, gamma=0.5):
    mu_n = (mu - mu.min()) / max(mu.max()-mu.min(), 1e-8)
    M = np.zeros((K,K))
    for i in range(K):
        for j in range(K):
            if i!=j: M[i,j] = gamma*(mu_n[j]-mu_n[i])
    return M


class AISODestroy:
    def __init__(self, dd, env, K=10, N=20, T=30):
        demands = env['demands']
        all_yards = sorted(env['all_yards'])
        yard_to_idx = {y:i for i,y in enumerate(all_yards)}
        Y = len(all_yards)

        feats, valid = [], []
        for i,d in enumerate(dd):
            if d is None: continue
            dem = demands[i]
            feats.append([yard_to_idx.get(dem.origin,0)/max(Y,1),
                          yard_to_idx.get(dem.dest,0)/max(Y,1),
                          math.log1p(dem.volume)/15.0,
                          1.0 if dem.commodity_type in DIRECT_ONLY_COMM else 0.0])
            valid.append(i)

        K = min(K, max(2, len(feats)//2))
        labels = AgglomerativeClustering(n_clusters=K, linkage='ward').fit(
                     np.array(feats, dtype=np.float32)).labels_

        self.cluster_of = [-1]*len(dd)
        self.cluster_demands = defaultdict(list)
        for j,i in enumerate(valid):
            c = int(labels[j])
            self.cluster_of[i] = c
            self.cluster_demands[c].append(i)
        self.valid  = valid
        self.K, self.N, self.T = K, N, T
        self.alpha, self.beta  = 0.4, 0.1
        mu = np.array([np.mean([demands[i].volume for i in self.cluster_demands.get(k,[0])])
                       for k in range(K)])
        self.M = build_smart_M(mu, K)
        self.W = np.random.dirichlet(np.ones(K), size=N)
        print(f'  AISO: K={K} clusters')

    def select(self, K_destroy, route_costs, temperature=1.0, noise=0.15):
        K,N,T = self.K, self.N, self.T
        valid  = self.valid
        if not valid: return []
        W = self.W + np.random.dirichlet(np.ones(K),size=N)*noise
        W /= W.sum(axis=1, keepdims=True)
        rc = np.zeros(max(len(route_costs), max(valid)+1))
        for i in valid: rc[i] = route_costs[i]
        rc_max = rc.max() or 1.0
        ac = np.argmax(W, axis=1)
        for _ in range(T):
            dists = [np.linalg.norm(W[i]-W[j]) for i in range(N) for j in range(i+1,N)]
            repul = 1+3*math.exp(-np.mean(dists)/0.12 if dists else 0)
            Meff  = self.M.copy(); Meff[Meff<0] *= repul
            for i in range(N):
                sj = np.array([float(W[i]@Meff@W[j])*(
                        np.mean([rc[x] for x in self.cluster_demands.get(ac[j],[valid[0]])])/rc_max)
                        if j!=i else -np.inf for j in range(N)])
                bj = int(np.argmax(sj)); cij = float(sj[bj])
                nW  = np.clip(W[i]+0.4*cij*(W[bj]-W[i]), 1e-8, None); nW/=nW.sum()
                nc2 = int(np.argmax(nW))
                if (np.mean([rc[x] for x in self.cluster_demands.get(nc2,valid)])/rc_max >=
                    np.mean([rc[x] for x in self.cluster_demands.get(ac[i],valid)])/rc_max):
                    W[i]=nW; ac[i]=nc2
                    mix=((1-self.beta)*W[i]+self.beta*W[bj]); W[i]=mix/mix.sum()
        self.W = 0.7*self.W + 0.3*W; self.W/=self.W.sum(axis=1,keepdims=True)
        cw = W.mean(axis=0)
        va  = np.array(valid)
        sc  = np.array([cw[self.cluster_of[i]]*rc[i] for i in valid])
        sc  = sc/(sc.max()+1e-8)
        pr  = np.exp(sc/max(temperature,0.01)); pr/=pr.sum()
        return va[np.random.choice(len(va), size=min(K_destroy,len(va)), replace=False, p=pr)].tolist()


# ── Hub destroy ────────────────────────────────────────────────────────────
def hub_destroy(dd, env, solver, K_hubs=3):
    """
    Find the K_hubs hubs with highest total transport cost contribution.
    Destroy (re-assign) all demands currently routed through those hubs.
    """
    hub_cost  = defaultdict(float)
    hub_dems  = defaultdict(list)
    route_idx = list(solver.get_routes())

    for i, d in enumerate(dd):
        if d is None: continue
        ri    = route_idx[i]
        route, cost = d['candidates'][ri]
        if route and len(route) == 2:   # via hub
            hub = route[0][1]
            hub_cost[hub]  += cost
            hub_dems[hub].append(i)

    if not hub_cost: return []
    worst_hubs = sorted(hub_cost, key=hub_cost.get, reverse=True)[:K_hubs]
    destroyed  = []
    for h in worst_hubs:
        destroyed.extend(hub_dems[h])
    return list(set(destroyed))


# ── CG-enhanced repair ─────────────────────────────────────────────────────
def cg_repair(destroyed, dd, env, current_routings, repair_time=5.0):
    """
    Re-price destroyed demands over ALL hubs, then use OR-Tools CP-SAT
    to jointly optimise their re-assignment.
    Key difference from v2: uses CG full-hub pricing instead of precomputed list.
    """
    if not destroyed:
        return list(current_routings)

    demands = env['demands']
    s       = env['settings']
    tc      = float(s.get('transport_cost_coefficient', 1.0))
    ic      = float(s.get('interchange_cost', 100))
    M_pen   = float(s.get('stress_penalty_M', 5))
    net     = env['net']

    # Build fresh full candidate lists for destroyed demands
    fresh_cands = {}
    for i in destroyed:
        dem   = demands[i]
        cands = [(None, M_pen*dem.volume*dem.sp_dist)]
        od_d  = net.dist(dem.origin, dem.dest)
        if not math.isinf(od_d):
            cands.append(([(dem.origin, dem.dest)],
                          tc*dem.volume*od_d + ic*dem.volume*net.interchanges(dem.origin, dem.dest)))
        if dem.commodity_type not in DIRECT_ONLY_COMM:
            for hub in env['all_yards']:
                if hub in (dem.origin, dem.dest): continue
                d1 = net.dist(dem.origin, hub)
                d2 = net.dist(hub, dem.dest)
                if math.isinf(d1) or math.isinf(d2): continue
                hc   = env['yard_info'].get(hub,{}).get('handling_cost',0.0)
                cost = (tc*dem.volume*(d1+d2)
                        + ic*dem.volume*(net.interchanges(dem.origin,hub)+net.interchanges(hub,dem.dest))
                        + hc*dem.volume)
                cands.append(([(dem.origin,hub),(hub,dem.dest)], cost))
        fresh_cands[i] = cands
        # Also extend dd[i] with any new candidates
        if dd[i]:
            existing = {str(r) for r,_ in dd[i]['candidates']}
            for rc in cands:
                if str(rc[0]) not in existing:
                    dd[i]['candidates'].append(rc)

    # OR-Tools CP-SAT over fresh candidates
    model  = cp_model.CpModel()
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = repair_time
    solver.parameters.num_search_workers  = 2
    scale  = 1000

    x_vars, cost_terms = {}, []
    for i in destroyed:
        cands  = fresh_cands[i]
        costs  = [int(c*scale) for _,c in cands]
        x      = model.NewIntVar(0, len(cands)-1, f'x{i}')
        cv     = model.NewIntVar(min(costs), max(costs), f'cv{i}')
        model.AddElement(x, costs, cv)
        x_vars[i] = (x, cands)
        cost_terms.append(cv)
    model.Minimize(sum(cost_terms))

    repaired = list(current_routings)
    status   = solver.Solve(model)
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        for i, (x, cands) in x_vars.items():
            repaired[i] = cands[solver.Value(x)][0]
    return repaired


# ── Perturbation + mini-CG ─────────────────────────────────────────────────
def perturb_cg(dd, env, best_routings, perturb_frac=0.15, cg_time=30):
    """
    Randomly reassign perturb_frac of demands, then run mini-CG.
    Escapes deep local optima.
    """
    n        = len(dd)
    perturbed = list(best_routings)
    valid     = [i for i,d in enumerate(dd) if d is not None]
    k         = max(1, int(len(valid) * perturb_frac))
    to_perturb = random.sample(valid, k)

    for i in to_perturb:
        d = dd[i]
        if d:
            perturbed[i] = random.choice(d['candidates'])[0]

    # Mini-CG from perturbed state
    new_routings = cg_solve(env, dd, max_iter=10, time_limit_s=cg_time, verbose=False)
    return new_routings


print('Operators ready.')

## 6. Main Loop (v3)

In [ ]:
def cg_v3_train(
    env, dd,
    n_episodes      = 150,
    sa_iters        = 10000,
    T0_frac         = 0.15,
    T_final_frac    = 0.0005,
    K_destroy_min   = 10,
    K_destroy_max   = 80,
    K_stag_step     = 5,    # K grows by this every stagnation period
    stag_window     = 10,   # episodes to consider "stagnant"
    K_hubs          = 3,
    K_clusters      = 10,
    aiso_noise      = 0.15,
    aiso_temp_init  = 2.0,
    aiso_temp_final = 0.3,
    perturb_every   = 30,   # episodes between perturbation restarts
    perturb_frac    = 0.15,
    repair_time     = 5.0,
    hub_destroy_every= 5,   # every N episodes, use hub destroy instead of AISO
    final_sa_iters  = 500000,
    cg_max_iter     = 50,
    cg_time_limit   = 180,
    print_every     = 25,
):
    t_total = time.time()

    # ── Step 1: Initial CG ────────────────────────────────────────────────
    print('\n── CG (initial) ──')
    cg_routings = cg_solve(env, dd, max_iter=cg_max_iter,
                           time_limit_s=cg_time_limit, verbose=True)
    cpp = build_cpp_solver(env, dd, cg_routings, verbose=True)

    greedy_sol   = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                    env['settings'], env['yard_info'])
    greedy_score, _ = evaluate(greedy_sol, env['net'], env['od_matrix'],
                               env['settings'], env['yard_info'])
    cg_score = cpp.get_score()
    if cg_score > greedy_score * 1.01:
        print(f'  CG worse than greedy, using greedy')
        cpp = build_cpp_solver(env, dd, greedy_sol.routings)
    else:
        print(f'  CG={cg_score:,.0f}  greedy={greedy_score:,.0f}  '
              f'gain={100*(greedy_score-cg_score)/greedy_score:.1f}%')

    # ── Step 2: AISO init ─────────────────────────────────────────────────
    aiso = AISODestroy(dd, env, K=K_clusters)

    best_score  = cpp.get_score()
    best_routes = list(cpp.get_routes())
    T0          = best_score * T0_frac
    T_end       = best_score * T_final_frac
    K_destroy   = K_destroy_min
    stag_count  = 0

    print(f'\n── v3 LNS+SA ──  init={best_score:,.0f}  ep={n_episodes}')
    print(f'  T0={T0:,.0f}  K={K_destroy_min}→{K_destroy_max}  hub_every={hub_destroy_every}  perturb_every={perturb_every}')

    for ep in range(1, n_episodes+1):
        frac     = ep / n_episodes
        T_ep     = T0 * ((T_end/T0) ** frac)
        aiso_tmp = aiso_temp_init * ((aiso_temp_final/aiso_temp_init) ** frac)

        # ── Adaptive K: grow when stagnating ─────────────────────────────
        K_destroy = min(K_destroy_max, K_destroy_min + stag_count // stag_window * K_stag_step)

        # ── Choose destroy operator ───────────────────────────────────────
        if ep % perturb_every == 0:
            # Perturbation + mini-CG restart
            perturbed_routings = perturb_cg(dd, env, get_routings(cpp, dd),
                                            perturb_frac=perturb_frac, cg_time=30)
            cpp2 = build_cpp_solver(env, dd, perturbed_routings)
            if cpp2.get_score() < best_score:
                cpp = cpp2
                print(f'  ep {ep:4d} [PERTURB+CG] → {cpp.get_score():,.0f} ✓')
            else:
                cpp.set_routes(best_routes)  # revert to best
            stag_count = 0
            continue

        elif ep % hub_destroy_every == 0:
            # Hub destroy
            destroyed = hub_destroy(dd, env, cpp, K_hubs=K_hubs)
        else:
            # AISO destroy
            route_idx   = list(cpp.get_routes())
            route_costs = np.array([dd[i]['candidates'][route_idx[i]][1]
                                    if dd[i] else 0.0 for i in range(len(dd))])
            destroyed = aiso.select(K_destroy, route_costs,
                                    temperature=aiso_tmp, noise=aiso_noise)

        # ── C++ SA ───────────────────────────────────────────────────────
        weights = [1e-6] * len(dd)
        for i in destroyed:
            weights[i] = 1.0 / max(len(destroyed), 1)
        sa = cpp.sa_run(weights, float(T_ep), float(T_ep*0.01),
                        int(sa_iters), int(sa_iters//10), int(ep))
        cpp.set_routes(sa.best_routes)

        # ── CG-enhanced repair ────────────────────────────────────────────
        if destroyed:
            repaired = cg_repair(destroyed, dd, env, get_routings(cpp, dd),
                                 repair_time=repair_time)
            cpp2 = build_cpp_solver(env, dd, repaired)
            if cpp2.get_score() < cpp.get_score():
                cpp = cpp2

        score = cpp.get_score()
        if score < best_score:
            best_score  = score
            best_routes = list(cpp.get_routes())
            stag_count  = 0
        else:
            stag_count += 1

        if ep % print_every == 0:
            print(f'  ep {ep:4d} | {score:>16,.0f} | best={best_score:>16,.0f} '
                  f'| K={K_destroy} stag={stag_count} T={T_ep:.0f}')

    # ── Step 3: Final SA ──────────────────────────────────────────────────
    print(f'\n── Final SA ({final_sa_iters:,}) ──')
    cpp.set_routes(best_routes)
    uw  = [1/max(len(dd),1)]*len(dd)
    pol = cpp.sa_run(uw, float(best_score*0.10), float(best_score*0.0001),
                     int(final_sa_iters), int(final_sa_iters//10), 999)
    if pol.best_score < best_score:
        best_score  = pol.best_score
        best_routes = list(pol.best_routes)
        print(f'  Improved: {best_score:,.0f}')
    else:
        print(f'  No improvement. best={best_score:,.0f}')

    cpp.set_routes(best_routes)
    print(f'  Total: {time.time()-t_total:.1f}s')
    return get_routings(cpp, dd), best_score


print('Training loop ready.')

## 7. Run

In [ ]:
CASE_ORDER = [
    ('l1',0.5),('l1',1.0),('l1',2.0),
    ('l2',0.5),('l2',1.0),('l2',2.0),
    ('l3',0.5),('l3',1.0),('l3',2.0),
]

HP = {
    'l1': dict(n_episodes=200, sa_iters=5000,
               K_destroy_min=5,  K_destroy_max=30,  K_stag_step=3,
               K_hubs=2, K_clusters=8,
               perturb_every=40, perturb_frac=0.15,
               hub_destroy_every=5, repair_time=3.0,
               cg_max_iter=50, cg_time_limit=60,
               final_sa_iters=300000),
    'l2': dict(n_episodes=120, sa_iters=10000,
               K_destroy_min=20, K_destroy_max=100, K_stag_step=10,
               K_hubs=3, K_clusters=10,
               perturb_every=30, perturb_frac=0.15,
               hub_destroy_every=5, repair_time=5.0,
               cg_max_iter=30, cg_time_limit=120,
               final_sa_iters=500000),
    'l3': dict(n_episodes=60,  sa_iters=20000,
               K_destroy_min=50, K_destroy_max=200, K_stag_step=20,
               K_hubs=5, K_clusters=15,
               perturb_every=20, perturb_frac=0.15,
               hub_destroy_every=5, repair_time=8.0,
               cg_max_iter=10, cg_time_limit=180,
               final_sa_iters=1000000),
}

solutions = {}
print('HP set.')

In [ ]:
for mult in [0.5, 1.0, 2.0]:
    tag = f'l1_{mult}'
    print(f'\n{"="*60}\n  L1 × {mult}\n{"="*60}')
    t0  = time.time()
    env = load_env('l1', mult)
    dd  = precompute(env)
    best_r, best_s = cg_v3_train(env, dd, **HP['l1'])
    sol = Solution(env['demands'], best_r)
    sc, stats = evaluate(sol, env['net'], env['od_matrix'], env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')
    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    print(f'  {time.time()-t0:.1f}s')

In [ ]:
for mult in [0.5, 1.0, 2.0]:
    tag = f'l2_{mult}'
    print(f'\n{"="*60}\n  L2 × {mult}\n{"="*60}')
    t0  = time.time()
    env = load_env('l2', mult)
    dd  = precompute(env)
    best_r, best_s = cg_v3_train(env, dd, **HP['l2'])
    sol = Solution(env['demands'], best_r)
    sc, stats = evaluate(sol, env['net'], env['od_matrix'], env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')
    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    print(f'  {time.time()-t0:.1f}s')

In [ ]:
for mult in [0.5, 1.0, 2.0]:
    tag = f'l3_{mult}'
    print(f'\n{"="*60}\n  L3 × {mult}\n{"="*60}')
    t0  = time.time()
    env = load_env('l3', mult)
    dd  = precompute(env)
    best_r, best_s = cg_v3_train(env, dd, **HP['l3'])
    sol = Solution(env['demands'], best_r)
    sc, stats = evaluate(sol, env['net'], env['od_matrix'], env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')
    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    print(f'  {time.time()-t0:.1f}s')

## 8. Submit

In [ ]:
rows = [['ID','data']]
for cid, (layer, mult) in enumerate(CASE_ORDER):
    tag = f'{layer}_{mult}'
    sol = solutions.get(tag)
    if sol is None:
        print(f'[{cid}] {tag} MISSING'); rows.append([cid,'{}']); continue
    ds = json.dumps(sol, separators=(',',':'))
    print(f'[{cid}] {tag}  {len(ds)/1e6:.1f}MB')
    rows.append([cid, ds])

with open('submission_v3.csv','w',newline='',encoding='utf-8') as f:
    csv.writer(f).writerows(rows)
print(f'Total: {sum(len(r[1]) for r in rows[1:])/1e6:.1f}MB')

from google.colab import files
files.download('submission_v3.csv')